# 可编辑的多叉树规则挖掘

`MultiwayTreeExtractor` 直接训练多叉树，支持数值区间、类别分组、逐节点修改、继续生长和剪枝。
本例用真实放款数据，以 FPD 为目标，并按 MOB1 的 7/3/0 天口径输出金额规则报告。
图形复用现有 `plot_tree` 风格。最后演示多分类节点分布。

卡方分裂参考 SAS Enterprise Miner 的 LogWorth；本实现使用 Bonferroni 调整，不声称与 SAS 算法完全相同。
[算法与编辑语义](../docs/multiway_tree.rst)。修改库代码后请重启内核从头运行。

In [ ]:
from pathlib import Path
import sys
import copy

ROOT = Path.cwd() if (Path.cwd() / 'hscredit').exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from hscredit import MultiwayTreeExtractor, plot_tree
from hscredit.utils import style_rule_table

output_dir = ROOT / 'examples/model_report/multiway_tree'
output_dir.mkdir(parents=True, exist_ok=True)
data = pd.read_excel(ROOT / 'examples/hscredit_yyp.xlsx')
features = ['衡枢鉴真分老客版', '近六个月非银多头机构数', '青云24', '商品类别']
train, test = train_test_split(data, test_size=0.3, random_state=42, stratify=data['FPD'])
print(f'总样本 {len(data)}，训练 {len(train)}，测试 {len(test)}')

## 1. 自动生成多叉树
`max_branches=4` 是上限，实际分支数由候选分裂价值和样本约束决定。

In [ ]:
tree = MultiwayTreeExtractor(
    target='FPD', features=features, max_branches=4, max_depth=2,
    min_samples_leaf=20, criterion='chi2', alpha=0.2,
    feature_map={'衡枢鉴真分老客版': '老客鉴真风险分'},
).fit(train)
display(tree.get_split_candidates(node=0))
display(tree.get_node_table())
fig = plot_tree(tree, save=str(output_dir / '自动多叉树.png'))
baseline = tree.evaluate({'训练集': train, '测试集': test})
display(baseline)

## 2. 人工指定三个阈值，直接生成四个分支
区间为左开右闭，缺失值可指定进入某个分支。操作会替换当前节点后代，其他子树不变。

In [ ]:
edited = copy.deepcopy(tree)
edited.manual_split(feature=features[0], thresholds=[0.12, 0.18, 0.25], node=0, missing_branch=0)
assert len(edited.nodes_[0].children) == 4
fig = plot_tree(edited, save=str(output_dir / '手工四叉树.png'))
display(style_rule_table(edited.get_rule_table(leaf_only=True)))

## 3. 调整阈值、继续分裂、剪枝
节点编号从 `nodes_` 或节点表中读取，不假定编号连续。人工空分支保留 0 样本，预测继承非空祖先概率。

In [ ]:
edited.set_thresholds(0, [0.10, 0.20, 0.30], missing_branch=0)
child = max(edited.nodes_[0].children, key=lambda node: edited.nodes_[node].counts.sum())
edited.grow(node=child)
fig = plot_tree(edited, save=str(output_dir / '继续生长.png'))
display(edited.get_node_table())
edited.prune(node=child)
assert not edited.nodes_[child].children
fig = plot_tree(edited, save=str(output_dir / '节点剪枝.png'))

## 4. 类别分组也可以人工调整
未知类别跟随保存的缺失分支；下面用商品类别演示完整分组。

In [ ]:
category_tree = copy.deepcopy(tree)
categories = data['商品类别'].dropna().unique().tolist()
train_categories = train['商品类别'].dropna().unique().tolist()
if len(train_categories) >= 2:
    groups = [[value] for value in train_categories[:3]]
    if len(train_categories) > 3:
        groups.append(train_categories[3:])
    category_tree.manual_split(feature='商品类别', category_groups=groups, node=0, missing_branch=0)
    display(category_tree.get_node_table())
    fig = plot_tree(category_tree, save=str(output_dir / '类别多叉树.png'))

## 5. 规则集与模型效果
全叶规则互斥且覆盖全部样本。业务拒绝策略应明确挑选风险叶节点，而不是将所有叶子做 OR。

In [ ]:
leaf_table = edited.report(test, leaf_only=True)
display(style_rule_table(leaf_table))
display(pd.concat([baseline.assign(模型='自动多叉树'), edited.evaluate({'训练集': train, '测试集': test}).assign(模型='人工调整树')], ignore_index=True))

rules = edited.get_rules()
masks = np.column_stack([rule.predict(test).to_numpy() for rule in rules])
assert np.all(masks.sum(axis=1) == 1)
assert leaf_table['样本总数'].sum() == len(test)
for rule in rules:
    print(rule.name, rule.expr)

risk_node = int(leaf_table.sort_values('坏样本率', ascending=False).iloc[0]['节点编号'])
strategy = edited.to_ruleset(nodes=[risk_node])
matched, details = strategy.evaluate(test)
print(f'选定风险叶节点 {risk_node}，命中 {matched.sum()} 人')
display(edited.report(test, overdue=['MOB1'], dpds=[7, 3, 0], amount='放款金额', leaf_only=True))

## 6. 保存模型与三种绘图后端
模型包含类别顺序、人工阈值和缺失/未知类别归属。只加载可信来源的模型文件。

In [ ]:
model_path = output_dir / '多叉树.pkl'
edited.save(model_path)
restored = MultiwayTreeExtractor.load(model_path)
np.testing.assert_allclose(restored.predict_proba(test), edited.predict_proba(test))
chart = plot_tree(restored, backend='pyecharts', save=str(output_dir / '多叉树.html'))
dot = plot_tree(restored, backend='graphviz')
(output_dir / '多叉树.gv').write_text(dot.source, encoding='utf-8')
print(f'模型与图形已保存到 {output_dir}')

## 7. 多分类节点分布
多分类节点逐类显示样本数和占比，模型评估提供准确率、交叉熵和 OVR AUC。

In [ ]:
status_data = pd.DataFrame({
    '账龄分组': ['短', '中', '长'] * 40,
    '账户状态': ['正常', '已结清', '逾期'] * 40,
})
status_tree = MultiwayTreeExtractor(target='账户状态', max_branches=4, max_depth=2).fit(status_data)
assert len(status_tree.nodes_[0].children) == 3
fig = plot_tree(status_tree, save=str(output_dir / '多分类三叉树.png'))
display(status_tree.evaluate())
display(status_tree.get_rule_table(leaf_only=True))